Day 3
Tasks:
Install and test core libraries: NLTK, spaCy, Hugging Face Transformers & Datasets, OpenCV, torchvision
Git workflow training: branching, commits, pull requests, basic code review etiquette
Download Flickr8k dataset (images + captions file) and load into a notebook
Deliverable: requirements.txt/environment.yml committed; first PR merged (even if trivial, to practice the workflow); dataset successfully loaded and previewed



Day 4
Tasks:
Text preprocessing on captions using NLTK/spaCy: tokenization, lowercasing, stop-word handling, building a vocabulary
Image preprocessing using OpenCV/torchvision: resizing, normalization, checking image channel consistency
Deliverable: Preprocessing notebook/script producing a cleaned, tokenized caption set and a resized/normalized image set (saved to data/processed/)

### 1. Environment Setup
Install required libraries. Since libraries are already downloaded, we run pip install to make them available in the Jupyter environment.

In [3]:
pip install nltk spacy transformers datasets opencv-python torchvision torch tqdm requests jupyter

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 2. Imports and NLTK Setup

In [5]:
import os
import cv2
import nltk
import spacy
import string
import torch
import torchvision.transforms as transforms
from collections import Counter
from tqdm import tqdm
from nltk.corpus import stopwords

# Download necessary NLTK data
nltk.download('punkt')
nltk.download('stopwords')

# Load spaCy model
# nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\umer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\umer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### 3. Text Preprocessing (Captions)
Tokenization, lowercasing, stop-word handling, and vocabulary building.

In [6]:
captions_path = '../data/captions.txt'
with open(captions_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

print(f"Total lines: {len(lines)}")
print("Sample lines:")
for line in lines[1:6]: # Skip header
    print(line.strip())

stop_words = set(stopwords.words('english'))
processed_captions = {}
vocab = Counter()

for line in tqdm(lines[1:]): # assuming first line is header
    parts = line.strip().split(',', 1)
    if len(parts) < 2:
        continue
    img_name, caption = parts[0], parts[1]
    
    # Lowercase
    caption = caption.lower()
    
    # Tokenize using NLTK
    tokens = nltk.word_tokenize(caption)
    
    # Remove punctuation and stop words
    tokens = [t for t in tokens if t not in string.punctuation and t not in stop_words]
    
    vocab.update(tokens)
    
    if img_name not in processed_captions:
        processed_captions[img_name] = []
    processed_captions[img_name].append(tokens)

print(f"Vocabulary size: {len(vocab)}")
print(f"Most common words: {vocab.most_common(10)}")

Total lines: 40456
Sample lines:
1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set of stairs in an entry way .
1000268201_693b08cb0e.jpg,A girl going into a wooden building .
1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .
1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playhouse .
1000268201_693b08cb0e.jpg,A little girl in a pink dress going into a wooden cabin .


100%|██████████| 40455/40455 [00:07<00:00, 5146.61it/s]

Vocabulary size: 8789
Most common words: [('dog', 8136), ('man', 7265), ('two', 5638), ('white', 3940), ('black', 3832), ('boy', 3581), ('woman', 3402), ('girl', 3328), ('wearing', 3062), ('people', 2883)]


### 4. Image Preprocessing
Resizing, normalization, and saving as tensors.

In [8]:
import numpy as np

img_dir = '../data/Images'
processed_dir = '../data/processed'
os.makedirs(processed_dir, exist_ok=True)
os.makedirs(os.path.join(processed_dir, 'images'), exist_ok=True)

# Define torchvision transforms
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

image_files = [f for f in os.listdir(img_dir) if f.endswith('.jpg') or f.endswith('.png')]
print(f"Total images found: {len(image_files)}")

# Processing a batch to test pipeline
sample_images = image_files[:8091] # Adjust this limit as needed

for img_name in tqdm(sample_images):
    img_path = os.path.join(img_dir, img_name)
    # Read image using OpenCV
    img = cv2.imread(img_path)
    
    if img is None:
        print(f"Error reading image: {img_name}")
        continue
        
    # OpenCV loads in BGR, convert to RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Check channel consistency
    if img.shape[2] != 3:
        print(f"Skipping {img_name} due to channel inconsistency: {img.shape}")
        continue
        
    # Apply torchvision transforms
    img_tensor = transform(img)
    
    # Save the processed tensor
    save_path = os.path.join(processed_dir, 'images', img_name.replace('.jpg', '.pt'))
    torch.save(img_tensor, save_path)

print("Sample image preprocessing complete.")

Total images found: 8091


100%|██████████| 8091/8091 [05:37<00:00, 23.97it/s]

Sample image preprocessing complete.
